# Mock 5B · Freeway Traffic Forecasting · Dự báo lưu lượng xe trên cao tốc (Time Series · Chuỗi thời gian)

| | 🇬🇧 English | 🇻🇳 Tiếng Việt |
|---|---|---|
| **Task** | 814 loop detectors on ten directional freeway panels of Los Angeles and Orange County (I-10, I-210, I-405, I-5) count the vehicles passing every 5 minutes. Each test **window** gives you the last **2 hours** of every detector; forecast each detector's flow **30 minutes after** the window's last time step. | 814 cảm biến vòng từ trên mười đoạn cao tốc một chiều ở Los Angeles và Orange County (I-10, I-210, I-405, I-5) đếm số xe đi qua mỗi 5 phút. Mỗi **cửa sổ** test cho bạn **2 giờ** dữ liệu gần nhất của mọi cảm biến; hãy dự báo lưu lượng của từng cảm biến **30 phút sau** bước thời gian cuối của cửa sổ. |
| **Data** | `train/train.csv`: the full panel for 1 Jan – 30 Apr (34,560 time steps × 814 sensors, one column per sensor) + `sensors.csv` (panel, freeway, direction, lanes, position along the panel). `public_test/` = 186 windows in **May** (151,366 rows to predict), `private_test/` = 180 windows in **June** (146,406 rows). | `train/train.csv`: toàn bộ dữ liệu 1/1 – 30/4 (34.560 bước thời gian × 814 cảm biến, mỗi cảm biến một cột) + `sensors.csv` (đoạn đường, cao tốc, hướng, số làn, vị trí dọc đoạn). `public_test/` = 186 cửa sổ trong **tháng 5** (151.366 dòng cần dự báo), `private_test/` = 180 cửa sổ trong **tháng 6** (146.406 dòng). |
| **Metric** | Mean absolute error (MAE) in vehicles per 5 minutes, **lower is better**. Ranking uses the **private** score. | Sai số tuyệt đối trung bình (MAE), đơn vị xe / 5 phút, **càng thấp càng tốt**. Xếp hạng theo điểm **private**. |
| **Submission** | `public_submission.csv`, `private_submission.csv` with columns `id,flow` (non-negative numbers), one row per row of the test file | `public_submission.csv`, `private_submission.csv` gồm cột `id,flow` (số không âm), mỗi dòng của file test một dòng |
| **Rules** | PyTorch model, use only the files provided (any of them, including other windows). **Honour code: do not download or use the original LargeST / Caltrans PeMS data** — the test months are public there. Calendar knowledge (US public holidays) is fine. | Mô hình PyTorch, chỉ dùng các file được cung cấp (file nào cũng được, kể cả các cửa sổ khác). **Giữ danh dự: không tải hay dùng dữ liệu gốc LargeST / Caltrans PeMS** — các tháng test có công khai ở đó. Được dùng kiến thức về lịch (ngày lễ của Mỹ). |

$$\text{MAE}=\frac{1}{N}\sum_{i=1}^{N}\lvert y_i-\hat y_i\rvert$$

**Files · Các file:**
- `train.csv`: `timestamp` (local time, every 5 minutes · giờ địa phương, mỗi 5 phút), then one column per sensor `S001 … S814` = vehicles counted in those 5 minutes over all lanes · số xe đếm được trong 5 phút đó trên mọi làn. An empty cell = no data · ô trống = không có dữ liệu.
- `sensors.csv`: `sensor_id, panel, freeway, direction, district, county, lanes, position_km, lat, lng`. Inside a panel the sensors are listed **in the direction of travel** (upstream → downstream), `position_km` = distance from the panel's first sensor · trong mỗi đoạn, cảm biến được xếp **theo chiều xe chạy** (thượng lưu → hạ lưu).
- `public_history.csv`, `private_history.csv`: `window_id, timestamp, S001 …` — 24 rows (2 hours) per window · 24 dòng (2 giờ) mỗi cửa sổ.
- `public_test.csv`, `private_test.csv`: `id, window_id, sensor_id, origin, target_time` — `origin` = the window's last time step, `target_time` = origin + 30 min.

🇬🇧 Real data: flows rise and fall with rush hours, weekends and holidays, and accidents cause sudden drops. Traffic in May–June is not the same as in January–April. There is at least one hour between any target and the next released data, so the answers cannot be read off the files.
🇻🇳 Dữ liệu thật: lưu lượng lên xuống theo giờ cao điểm, cuối tuần và ngày lễ, còn tai nạn gây sụt giảm đột ngột. Giao thông tháng 5–6 không giống tháng 1–4. Giữa mỗi thời điểm cần dự báo và dữ liệu được công bố tiếp theo luôn cách ít nhất một giờ, nên không thể đọc đáp án từ các file.

Data · Dữ liệu: Caltrans PeMS via [LargeST](https://github.com/liuxu77/LargeST) (Liu et al., NeurIPS 2023), CC BY-NC 4.0 — subset, re-dated and re-labelled (see `dataset/README.md`, `dataset/LICENSE`).
Inspired by · Lấy cảm hứng từ: [IEEE BigData 2026 TrafficFlowBench (Kaggle)](https://www.kaggle.com/competitions/2026-ieee-big-data-traffic-flow-bench).

🇬🇧 This baseline runs end-to-end and submits a result. Beating it is up to you.
🇻🇳 Baseline này chạy trọn vẹn và nộp được kết quả. Vượt qua nó là việc của bạn.

## 1. Setup and Imports · Cài đặt và thư viện

In [ ]:
import os
import re
import shutil
import unicodedata
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration and Hyperparameters · Cấu hình và siêu tham số
🇬🇧 Nothing to fill in here: the data downloads itself, and your team comes from your account on the site when you submit (section 7).
🇻🇳 Không cần điền gì ở đây: dữ liệu tự tải về, và đội của bạn được lấy từ tài khoản trên trang web khi nộp bài (phần 7).

In [ ]:
# ---- Data / Dữ liệu ----
# The dataset downloads itself from GitHub straight onto this Colab session's own
# disk. No Google Drive and no mounting: reading thousands of small files over a
# Drive mount is what made the old flow slow, and local disk is several times
# faster per epoch.
# Bộ dữ liệu tự tải từ GitHub vào ổ đĩa của phiên Colab này. Không cần Drive.
TASK    = "5B"
FOLDER  = "5B_traffic_flow"
RELEASE = "https://github.com/ftulabs/olympiad-datasets/releases/download/mock-5"

# Already have the folder? Point this at the parent of 5B_traffic_flow/ and nothing is
# downloaded. / Đã có sẵn thư mục? Trỏ vào thư mục cha của 5B_traffic_flow/.
ROOT_OVERRIDE = ""

import os, subprocess, time

ROOT = ROOT_OVERRIDE or "/content/olympiad"
if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    os.makedirs(ROOT, exist_ok=True)
    zip_path = f"{ROOT}/{FOLDER}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {FOLDER}.zip / Đang tải ...", flush=True)
        t0 = time.time()
        subprocess.run(["curl", "-fsSL", "--retry", "3", "-o", zip_path + ".part",
                        f"{RELEASE}/{FOLDER}.zip"], check=True)
        os.replace(zip_path + ".part", zip_path)  # only a complete download gets the final name
        print(f"  {os.path.getsize(zip_path)/1e6:.0f} MB in {time.time()-t0:.0f}s", flush=True)
    print("Extracting / Đang giải nén ...", flush=True)
    subprocess.run(["unzip", "-nq", zip_path, "-d", ROOT], check=True)

if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    raise FileNotFoundError(
        f"{ROOT}/{FOLDER}/dataset is missing after the download. Re-run this cell. "
        f"/ Thiếu dữ liệu sau khi tải. Hãy chạy lại ô này."
    )
print(f"Data root / Thư mục dữ liệu: {ROOT}")

DATA_DIR = f"{ROOT}/5B_traffic_flow/dataset"
TRAIN_CSV = f"{DATA_DIR}/train/train.csv"
SENSORS_CSV = f"{DATA_DIR}/train/sensors.csv"
PUBLIC_HIST_CSV = f"{DATA_DIR}/public_test/public_history.csv"
PUBLIC_CSV = f"{DATA_DIR}/public_test/public_test.csv"
PRIVATE_HIST_CSV = f"{DATA_DIR}/private_test/private_history.csv"
PRIVATE_CSV = f"{DATA_DIR}/private_test/private_test.csv"
SAVE_DIR = "./checkpoints"

# ---- Hyperparameters / Siêu tham số ----
N_LAGS = 12  # last hour of history used as input / dùng 1 giờ gần nhất làm đầu vào
HORIZON = 6  # 6 x 5 min = 30 min ahead / dự báo trước 30 phút
N_TRAIN_ORIGINS = 3000  # random forecast origins sampled from train / số thời điểm dự báo lấy ngẫu nhiên từ train
VAL_FROM = "2030-04-17"  # last 2 weeks of train = validation / 2 tuần cuối của train = validation
BATCH_SIZE = 4096
EPOCHS = 3
LR = 2e-3
HIDDEN = 64
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

os.makedirs(SAVE_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3. Data Loading and Features · Đọc dữ liệu và tạo đặc trưng

In [ ]:
train_df = pd.read_csv(TRAIN_CSV, parse_dates=["timestamp"])
sensors = pd.read_csv(SENSORS_CSV)
SENSOR_IDS = sensors["sensor_id"].tolist()
public_hist = pd.read_csv(PUBLIC_HIST_CSV, parse_dates=["timestamp"])
private_hist = pd.read_csv(PRIVATE_HIST_CSV, parse_dates=["timestamp"])
public_df = pd.read_csv(PUBLIC_CSV, parse_dates=["origin", "target_time"])
private_df = pd.read_csv(PRIVATE_CSV, parse_dates=["origin", "target_time"])
print(f"Train: {len(train_df):,} time steps x {len(SENSOR_IDS)} sensors, "
      f"{train_df['timestamp'].min()} .. {train_df['timestamp'].max()}")
print(f"Public: {public_hist['window_id'].nunique()} windows, {len(public_df):,} rows | "
      f"Private: {private_hist['window_id'].nunique()} windows, {len(private_df):,} rows")

TARGET = "flow"
FLOW = train_df[SENSOR_IDS].to_numpy(dtype=np.float32)  # (time, sensor), NaN = no data / không có dữ liệu
SCALE = np.nanmean(FLOW, axis=0)  # each sensor's average flow / lưu lượng trung bình của từng cảm biến


def fill_gaps(block):
    """EN: forward-fill gaps along time, then use the sensor's average. VI: điền ô trống bằng giá trị trước đó, rồi bằng trung bình."""
    x = block.astype(np.float32)  # (time, sensor) copy
    for t in range(1, len(x)):
        x[t] = np.where(np.isnan(x[t]), x[t - 1], x[t])
    return np.where(np.isnan(x), SCALE, x)


def window_features(lags, target_time):
    """EN: inputs of one window for all sensors: last N_LAGS flows / sensor average + time of day + weekend flag.
    VI: đầu vào của một cửa sổ cho mọi cảm biến: N_LAGS giá trị gần nhất / trung bình + giờ trong ngày + cuối tuần."""
    x = (lags[-N_LAGS:] / SCALE).T  # (sensor, N_LAGS)
    minutes = target_time.hour * 60 + target_time.minute
    cal = [np.sin(2 * np.pi * minutes / 1440), np.cos(2 * np.pi * minutes / 1440), float(target_time.dayofweek >= 5)]
    return np.hstack([x, np.tile(cal, (len(SCALE), 1))]).astype(np.float32)


def sample_train(origins):
    """EN: windows cut from train at the given origins (row numbers); target = flow HORIZON steps later / SCALE.
    VI: cắt cửa sổ từ train tại các thời điểm cho trước; nhãn = lưu lượng sau HORIZON bước / SCALE."""
    X, Y = [], []
    for o in origins:
        X.append(window_features(fill_gaps(FLOW[o - N_LAGS + 1:o + 1]), train_df["timestamp"].iloc[o + HORIZON]))
        Y.append(FLOW[o + HORIZON] / SCALE)
    X, Y = np.concatenate(X), np.concatenate(Y)
    ok = ~np.isnan(Y)  # skip targets with no data / bỏ nhãn bị thiếu
    return X[ok], Y[ok]


# EN: time-based split: random origins before VAL_FROM for training, the last two weeks for validation
# VI: tách theo thời gian: thời điểm ngẫu nhiên trước VAL_FROM để huấn luyện, hai tuần cuối để kiểm tra
rng = np.random.default_rng(SEED)
val_start = int(np.searchsorted(train_df["timestamp"].to_numpy(), np.datetime64(VAL_FROM)))
tr_origins = rng.integers(N_LAGS, val_start - HORIZON, N_TRAIN_ORIGINS)
val_origins = rng.integers(val_start + N_LAGS, len(train_df) - HORIZON, 300)
X_train, y_train = sample_train(tr_origins)
X_val, y_val = sample_train(val_origins)
print(f"Training samples: {len(X_train):,} | Validation samples: {len(X_val):,}")


def test_features(hist, test_df):
    """EN: features for every (window, sensor) of a test file, in the order of its rows.
    VI: đặc trưng cho mọi (cửa sổ, cảm biến) của file test, theo đúng thứ tự các dòng."""
    sidx = {s: i for i, s in enumerate(SENSOR_IDS)}
    rows = []
    for wid, g in test_df.groupby("window_id", sort=False):
        h = hist[hist["window_id"] == wid]
        f = window_features(fill_gaps(h[SENSOR_IDS].to_numpy(dtype=np.float32)), g["target_time"].iloc[0])
        rows.append(pd.DataFrame(f[g["sensor_id"].map(sidx).to_numpy()], index=g.index))
    return pd.concat(rows).loc[test_df.index].to_numpy(dtype=np.float32).copy()  # writable


X_public = test_features(public_hist, public_df)
X_private = test_features(private_hist, private_df)
SCALE_PUBLIC = public_df["sensor_id"].map(dict(zip(SENSOR_IDS, SCALE))).to_numpy()
SCALE_PRIVATE = private_df["sensor_id"].map(dict(zip(SENSOR_IDS, SCALE))).to_numpy()
train_df.iloc[:3, :8]

## 4. Model Architecture · Kiến trúc mô hình

In [ ]:
class FlowMLP(nn.Module):
    """EN: last hour of the sensor's own flow + time of day -> flow in 30 minutes (relative to the sensor's average).
    VI: 1 giờ lưu lượng gần nhất của chính cảm biến + giờ trong ngày -> lưu lượng sau 30 phút (so với trung bình)."""

    def __init__(self, n_in, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, x):
        return self.net(x).squeeze(-1)


model = FlowMLP(X_train.shape[1]).to(DEVICE)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Training · Huấn luyện

In [ ]:
@torch.no_grad()
def predict_flow(model, X, scale):
    """EN: forecasts in vehicles per 5 minutes. VI: dự báo theo đơn vị xe / 5 phút."""
    model.eval()
    out = [model(torch.from_numpy(X[i:i + 65536]).to(DEVICE)).cpu().numpy() for i in range(0, len(X), 65536)]
    return np.clip(np.concatenate(out) * scale, 0, None)


train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)),
                          batch_size=BATCH_SIZE, shuffle=True)
scale_val = np.tile(SCALE, len(val_origins))[~np.isnan(np.concatenate([FLOW[o + HORIZON] for o in val_origins]))]
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_mae = float("inf")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)

    val_mae = float(np.mean(np.abs(predict_flow(model, X_val, scale_val) - y_val * scale_val)))
    msg = f"Epoch {epoch:02d} | Loss={total_loss / len(X_train):.4f} | Val MAE={val_mae:.2f} vehicles"
    if val_mae < best_mae:
        best_mae = val_mae
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pt"))
        msg += "  ✓ best model saved"
    print(msg)

print(f"\nBest validation MAE: {best_mae:.2f}")

## 6. Predict Public & Private Test · Dự đoán tập Public & Private
🇬🇧 Creates `public_submission.csv` / `private_submission.csv` (+ `.zip`), like the official competition.
🇻🇳 Tạo `public_submission.csv` / `private_submission.csv` (+ `.zip`), giống định dạng cuộc thi chính thức.

In [ ]:
def export(ids, preds, csv_path):
    """Save <id, flow> CSV and zip it. / Lưu dự báo ra CSV và nén zip."""
    pd.DataFrame({"id": ids, TARGET: preds}).to_csv(csv_path, index=False)
    zip_path = csv_path.replace(".csv", ".zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname=os.path.basename(csv_path))
    print(f"Saved {csv_path} and {zip_path}")


model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "best_model.pt"), map_location=DEVICE))
export(public_df["id"], predict_flow(model, X_public, SCALE_PUBLIC), "public_submission.csv")
export(private_df["id"], predict_flow(model, X_private, SCALE_PRIVATE), "private_submission.csv")
pd.read_csv("public_submission.csv").head()

## 7. Submit · Nộp bài

🇬🇧 Your team comes from your account on [seleccion.ftds.online](https://seleccion.ftds.online/), not from a name typed here. The cell below prints a short code once per Colab session; type it on the site and this notebook submits as you.

🇻🇳 Đội của bạn lấy từ tài khoản trên [seleccion.ftds.online](https://seleccion.ftds.online/). Ô bên dưới in ra một mã ngắn mỗi phiên Colab; nhập mã đó trên trang web.

In [ ]:
def check_submission(csv_path, test_df):
    """Validate the file format before submitting. / Kiểm tra định dạng file trước khi nộp."""
    sub = pd.read_csv(csv_path)
    assert list(sub.columns) == ["id", TARGET], f"{csv_path}: columns must be ['id', '{TARGET}']"
    assert len(sub) == len(test_df) and set(sub["id"]) == set(test_df["id"]), f"{csv_path}: ids do not match the test set"
    assert sub[TARGET].notna().all(), f"{csv_path}: contains missing values / có giá trị bị thiếu"
    assert np.isfinite(sub[TARGET]).all(), f"{csv_path}: flow must be finite numbers / lưu lượng phải là số hữu hạn"
    assert (sub[TARGET] >= 0).all(), f"{csv_path}: flow must be >= 0 / lưu lượng phải >= 0"


check_submission("public_submission.csv", public_df)
check_submission("private_submission.csv", private_df)

In [ ]:
# ===================================================================
#  Olympiad AI - submit.  Paste at the END of your notebook, then run.
#  No key to copy: the first run prints a code you enter on the site.
# ===================================================================
BASE    = "https://seleccion.ftds.online"
PROBLEM = "5B"          # which problem this notebook is for
CONTEST = "mock-5"    # this notebook is for Warmup Contest 5
PUBLIC  = "public_submission.csv"
PRIVATE = "private_submission.csv"
# -------------------------------------------------------------------
import os, time, requests

token = globals().get("_OLY_TOKEN")
if not token:
    start = requests.post(f"{BASE}/api/pair/start", timeout=30).json()
    print(f"\n  1. open   {start['enter_at']}")
    print(f"  2. enter  {start['code']}\n  waiting...", flush=True)
    for _ in range(int(start["expires_in"]) // 3):
        time.sleep(3)
        got = requests.get(f"{BASE}/api/pair/poll",
                           params={"code": start["code"]}, timeout=30).json()
        if got.get("ready"):
            token = globals()["_OLY_TOKEN"] = got["token"]
            print("  paired ✓ (this notebook stays paired for the session)")
            break
    else:
        raise SystemExit("pairing timed out - run this cell again")

missing = [f for f in (PUBLIC, PRIVATE) if not os.path.exists(f)]
if PUBLIC in missing:
    raise SystemExit(f"{PUBLIC} not found - run the cell that writes it first")

files = {"public": open(PUBLIC, "rb")}
if PRIVATE not in missing:
    files["private"] = open(PRIVATE, "rb")
try:
    r = requests.post(f"{BASE}/api/submit", headers={"X-Oly-Token": token},
                      data={"problem": PROBLEM, "contest": CONTEST},
                      files=files, timeout=600)
finally:
    for f in files.values():
        f.close()

try:
    d = r.json()
except ValueError:
    raise SystemExit(f"server {r.status_code}: {r.text[:300]}")

if r.status_code == 429:
    print(f"⏳ cooldown - wait {d.get('retry_after')}s and run again")
elif r.status_code == 401:
    globals().pop("_OLY_TOKEN", None)
    print(f"❌ {d.get('detail')}  (run this cell again to re-pair)")
elif r.status_code != 200:
    print(f"❌ {r.status_code}: {d.get('detail') or d.get('message')}")
elif d["public"]["status"] != "ok":
    # the grader's own message about your file - it says what to fix
    print(f"❌ {d['public']['status'].upper()}: {d['public']['message']}")
else:
    p, c = d["public"], (d.get("counting") or {})
    print(f"✅ {d['problem']} · {d['title']}")
    print(f"   {d['metric']} = {p['score']:.4f}   {p['tier'] or '(below the lowest tier)'}")
    print(f"   entry #{d['entries']}   ·   rank: {d['rank']}")
    if c.get("score") is not None and abs(c["score"] - p["score"]) > 1e-12:
        print(f"   counting entry: {c['score']:.4f} {c['tier']}")
    print(f"   {BASE}/")